In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1LIf3nVYWkz1fpXX5O8HPI0ey5By9WVt_", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_00_intro.mp3"))


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Structured Output & Prompt Chaining

*Part 4 of the Vizuara series on Prompt Design Principles*
*Estimated time: 60 minutes*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/prompt-design-principles/practice/4/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Why Does This Matter?

Here is a situation every engineer eventually faces: you build a prototype with an LLM, it works beautifully in a Jupyter notebook, and then you try to plug it into a real application. The LLM returns "Sure! Here is the JSON you requested:" followed by a code block with backticks, a trailing comma, and a helpful paragraph of commentary. Your `json.loads()` call explodes. Your pipeline crashes. Your users see an error page.

This is the **structured output problem** — and it is the single biggest gap between "LLM demo" and "LLM in production." If your model cannot reliably produce machine-parseable output, you cannot build systems on top of it.

But there is a second problem that is even more subtle. Suppose you fix the formatting issue and your LLM reliably outputs clean JSON. Now you want to build something more ambitious: a pipeline that reads a document, extracts entities, classifies each entity, and then writes a summary. Four LLM calls, chained together. If each step is 95% reliable, your overall pipeline is only **81.5% reliable**. At 85% per step, it drops to **52.2%**. Your chain is only as strong as its weakest link — actually, it is weaker than that.

In this notebook, we will:
- Master three approaches to getting structured output from LLMs
- Understand the mathematics of error propagation in chains
- Build multi-step prompt chains with validation gates
- Create a production-grade document analysis pipeline

Let us start with a concrete example of why unstructured output is dangerous.

In [ ]:
# Setup — install dependencies
!pip install -q anthropic matplotlib pandas numpy

import os
import json
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from anthropic import Anthropic

client = Anthropic()

def query_llm(system_prompt, user_message, model="claude-sonnet-4-20250514", max_tokens=1024):
    """Send a message to Claude and return the response text."""
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip()

print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition

### The Contract Between LLM and Code

Imagine you hire a freelance translator. If you just say "translate this document," they might email you back a Word file, a PDF, or just paste the translation into the email body with their invoice attached. All of these contain a valid translation — but if your automated system expects a plain text file with one sentence per line, none of them work.

**Structured output is a contract.** It is an agreement between the LLM (the producer) and your code (the consumer) about the exact format of the data exchange. JSON, XML, CSV — these are not just formats, they are protocols that make machine-to-machine communication possible.

Let us see the problem in action.

In [ ]:
#@title 🎧 Code Walkthrough: Unstructured Output Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_unstructured_output_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# The unstructured output problem
unstructured_prompt = "Extract the person's name, age, and city from this text."
text = "Maria Chen, a 34-year-old software engineer, recently moved to Seattle."

response = query_llm("", f"{unstructured_prompt}\n\nText: {text}")
print("Unstructured response:")
print(response)
print("\n" + "-" * 50)

# Try to use this programmatically... how?
# We would need fragile regex or string parsing.
# Different runs might produce different formats.
print("\nCan we reliably parse this? Let us try json.loads()...")
try:
    data = json.loads(response)
    print(f"Parsed: {data}")
except json.JSONDecodeError as e:
    print(f"FAILED: {e}")
    print("This is exactly the problem. The LLM gave us useful content")
    print("in a format our code cannot consume.")

In [ ]:
#@title 🎧 Code Walkthrough: Structured Output Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_structured_output_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Now compare this with a structured approach:

In [ ]:
# The structured output solution
structured_prompt = """Extract information from the text below.
Return ONLY a JSON object with these exact keys: name, age, city.
No explanation, no markdown, no extra text — just the JSON object.

Example:
Text: "John Smith, 28, lives in Boston."
{"name": "John Smith", "age": 28, "city": "Boston"}"""

text = "Maria Chen, a 34-year-old software engineer, recently moved to Seattle."
response = query_llm("", f"{structured_prompt}\n\nText: {text}")
print("Structured response:")
print(response)

# Now parse it
try:
    data = json.loads(response)
    print(f"\nParsed successfully!")
    print(f"  Name: {data['name']}")
    print(f"  Age:  {data['age']}")
    print(f"  City: {data['city']}")
except json.JSONDecodeError as e:
    print(f"Parse failed: {e}")

This is exactly what we want. The same information, but in a format our code can reliably consume.


In [ ]:
#@title 🎧 Listen: Why Chaining Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_why_chaining_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Why Chaining Matters

Now, structured output solves the format problem for a single LLM call. But real applications are rarely a single call. They are **pipelines** — sequences of LLM calls where the output of one step feeds into the next.

Think of it like a factory assembly line. If one station produces defective parts 5% of the time, a 4-station line produces defective products much more than 20% of the time — because a defect at any station ruins the final product.

### Think About This

You are building a customer complaint system with 3 steps:
1. Extract the complaint category (billing, shipping, product quality)
2. Look up relevant policies based on the category
3. Draft a personalized response

If Step 1 misclassifies the complaint, Steps 2 and 3 produce a perfectly written but completely wrong response. The customer gets a beautifully formatted answer about a billing policy when their package never arrived. Errors do not just accumulate — they **compound**.

In [ ]:
#@title 🎧 Listen: Math Error Propagation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_math_error_propagation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics

### Error Propagation in Prompt Chains

Let us formalize this intuition. Suppose we have a chain of $n$ LLM steps, each with success probability $p_i$. If we assume the steps are independent (a simplifying assumption we will revisit), the probability that the entire chain succeeds is:

$$P(\text{chain correct}) = \prod_{i=1}^{n} p_i$$

**What does this mean computationally?** Each step acts as a filter — the chain can only succeed if every single step succeeds. Even high per-step accuracy can lead to poor chain accuracy when the chain is long.

Let us plug in some concrete numbers.

**Scenario 1: Good per-step accuracy (95%)**

With 4 steps, each at 95% accuracy:

$$P(\text{chain}) = 0.95^4 = 0.95 \times 0.95 \times 0.95 \times 0.95 = 0.8145$$

So a chain of 4 "good" steps only gives us 81.5% end-to-end accuracy. Almost 1 in 5 runs fails.

**Scenario 2: Mediocre per-step accuracy (85%)**

With 4 steps, each at 85% accuracy:

$$P(\text{chain}) = 0.85^4 = 0.85 \times 0.85 \times 0.85 \times 0.85 = 0.5220$$

Now we are below coin-flip territory. More than half of all runs produce an incorrect final result. This is not a system anyone would deploy.

**Scenario 3: Mixed accuracy**

Suppose your 4 steps have accuracies of 0.98, 0.90, 0.95, and 0.88:

$$P(\text{chain}) = 0.98 \times 0.90 \times 0.95 \times 0.88 = 0.7370$$

Notice that the weakest step (0.88) drags down the entire chain. This tells us that in a prompt chain, you should invest the most effort in improving your **weakest** step — that is where the leverage is.

In [ ]:
#@title 🎧 What to Look For: Chain Reliability Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_chain_reliability_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualization: Chain reliability drops fast
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Chain accuracy vs number of steps for different per-step accuracies
step_counts = range(1, 11)
per_step_accuracies = [0.99, 0.95, 0.90, 0.85, 0.80]
colors = ['#2E7D32', '#4CAF50', '#FFC107', '#FF9800', '#F44336']

for acc, color in zip(per_step_accuracies, colors):
    chain_accs = [acc ** n for n in step_counts]
    axes[0].plot(step_counts, chain_accs, 'o-', color=color,
                 label=f'p = {acc:.0%}', linewidth=2, markersize=6)

axes[0].axhline(y=0.80, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
axes[0].set_xlabel('Number of Chain Steps', fontsize=12)
axes[0].set_ylabel('Chain Accuracy', fontsize=12)
axes[0].set_title('Chain Accuracy Drops Rapidly With More Steps', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Right panel: Compare specific scenarios (bar chart)
scenarios = ['95% x 4\nsteps', '85% x 4\nsteps', 'Mixed\n4 steps', '99% x 4\nsteps']
chain_values = [0.95**4, 0.85**4, 0.98*0.90*0.95*0.88, 0.99**4]
bar_colors = ['#4CAF50', '#F44336', '#FF9800', '#2E7D32']

bars = axes[1].bar(scenarios, chain_values, color=bar_colors, edgecolor='black', linewidth=1.2)
for bar, val in zip(bars, chain_values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

axes[1].set_ylabel('End-to-End Chain Accuracy', fontsize=12)
axes[1].set_title('Numerical Examples: Chain Error Propagation', fontsize=13)
axes[1].set_ylim(0, 1.15)
axes[1].axhline(y=0.80, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key insight: Even 95% per-step accuracy gives only 81.5% over 4 steps.")
print("At 85% per step, you are worse than a coin flip (52.2%).")
print("This is why gate checks between steps are essential.")

In [ ]:
#@title 🎧 Listen: How Gate Checks Help
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_how_gate_checks_help.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### How Gate Checks Help

The solution is to add **validation gates** between chain steps. A gate checks whether the output of step $i$ is valid before passing it to step $i+1$. If the gate fails, we can retry the step, use a fallback, or abort gracefully.

With a gate that catches $g$ fraction of errors, the effective per-step accuracy becomes:

$$p_i^{\text{gated}} = p_i + (1 - p_i) \cdot g$$

Let us see this numerically. If $p_i = 0.90$ and our gate catches 80% of errors ($g = 0.80$):

$$p_i^{\text{gated}} = 0.90 + (1 - 0.90) \times 0.80 = 0.90 + 0.08 = 0.98$$

Now a 4-step chain with gated steps:

$$P(\text{gated chain}) = 0.98^4 = 0.9224$$

Compare this with the ungated version: $0.90^4 = 0.6561$. The gates boosted chain accuracy from **65.6% to 92.2%**. This is exactly what we want.

In [ ]:
#@title 🎧 Transition: Three Approaches Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_three_approaches_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Three Approaches to Structured Output

Now let us build this practically. There are three main approaches to getting structured output from LLMs, each with different trade-offs. We will implement all three and compare them.

### Approach 1: Format Instructions in the Prompt

The simplest approach — just tell the model exactly what format you want.

In [ ]:
#@title 🎧 Code Walkthrough: Approach1 Format Instructions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_approach1_format_instructions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Approach 1: Format Instructions
def extract_with_format_instructions(text):
    """Extract structured data using explicit format instructions."""
    prompt = f"""Extract all entities from the text below.
Return ONLY a valid JSON object with these exact keys:
- "persons": list of person names (strings)
- "organizations": list of organization names (strings)
- "locations": list of location names (strings)
- "dates": list of date references (strings)

Rules:
- Return ONLY the JSON object, nothing else
- Use empty lists [] for categories with no entities
- Do not include explanations or markdown formatting

Text: {text}"""

    response = query_llm("You output only valid JSON.", prompt)

    # Clean up common issues
    response = response.strip()
    if response.startswith("```"):
        response = response.split("\n", 1)[1]  # Remove first line
        response = response.rsplit("```", 1)[0]  # Remove last fence
    response = response.strip()

    return json.loads(response)

# Test it
test_text = """On March 15, 2024, Satya Nadella announced that Microsoft would partner
with OpenAI to build a new research lab in London. The project, backed by
the Gates Foundation, aims to advance AI safety research by 2026."""

result = extract_with_format_instructions(test_text)
print("Extracted entities:")
print(json.dumps(result, indent=2))

In [ ]:
#@title 🎧 Code Walkthrough: Approach2 Xml Tags
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_approach2_xml_tags.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Approach 2: XML Tags as Delimiters

XML tags give the model clear start and end markers. This works especially well when you need to extract multiple fields with different formats.

In [ ]:
# Approach 2: XML-tagged structured output
def extract_with_xml_tags(text):
    """Extract structured data using XML tag delimiters."""
    prompt = f"""Analyze the text below. Return your analysis inside XML tags as shown.

<analysis>
  <summary>One-sentence summary of the text</summary>
  <entities>
    <person>Name1</person>
    <person>Name2</person>
    <organization>Org1</organization>
    <location>Location1</location>
  </entities>
  <sentiment>positive, negative, or neutral</sentiment>
  <confidence>high, medium, or low</confidence>
</analysis>

Only include entity tags for entities that actually appear in the text.
Do not include any text outside the <analysis> tags.

Text: {text}"""

    response = query_llm("", prompt)
    return response

test_text = """Tesla reported record quarterly earnings, driven by strong Model Y
sales in Europe and China. CEO Elon Musk credited the Berlin and
Shanghai factories for exceeding production targets."""

result = extract_with_xml_tags(test_text)
print("XML-tagged response:")
print(result)

In [ ]:
# Parse the XML response
import re

def parse_xml_analysis(response):
    """Parse an XML-tagged analysis response into a dictionary."""
    parsed = {}

    # Extract summary
    summary_match = re.search(r'<summary>(.*?)</summary>', response, re.DOTALL)
    parsed['summary'] = summary_match.group(1).strip() if summary_match else ""

    # Extract entities by type
    parsed['persons'] = re.findall(r'<person>(.*?)</person>', response)
    parsed['organizations'] = re.findall(r'<organization>(.*?)</organization>', response)
    parsed['locations'] = re.findall(r'<location>(.*?)</location>', response)

    # Extract sentiment and confidence
    sentiment_match = re.search(r'<sentiment>(.*?)</sentiment>', response)
    parsed['sentiment'] = sentiment_match.group(1).strip() if sentiment_match else ""

    confidence_match = re.search(r'<confidence>(.*?)</confidence>', response)
    parsed['confidence'] = confidence_match.group(1).strip() if confidence_match else ""

    return parsed

parsed = parse_xml_analysis(result)
print("\nParsed result:")
for key, value in parsed.items():
    print(f"  {key}: {value}")

In [ ]:
#@title 🎧 Code Walkthrough: Approach3 Schema In Prompt
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_approach3_schema_in_prompt.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Approach 3: Schema-in-Prompt (Constrained Output)

The most robust approach: embed the exact JSON schema in your prompt so the model knows exactly what structure to produce.

In [ ]:
# Approach 3: JSON Schema in prompt
ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        "persons": {"type": "array", "items": {"type": "object", "properties": {
            "name": {"type": "string"},
            "role": {"type": "string"}
        }}},
        "organizations": {"type": "array", "items": {"type": "object", "properties": {
            "name": {"type": "string"},
            "type": {"type": "string", "enum": ["company", "nonprofit", "government", "academic"]}
        }}},
        "locations": {"type": "array", "items": {"type": "object", "properties": {
            "name": {"type": "string"},
            "type": {"type": "string", "enum": ["city", "country", "region", "facility"]}
        }}},
        "key_facts": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["persons", "organizations", "locations", "key_facts"]
}

def extract_with_schema(text):
    """Extract structured data by providing the JSON schema in the prompt."""
    schema_str = json.dumps(ENTITY_SCHEMA, indent=2)

    prompt = f"""Extract structured information from the text below.

Your response must be a valid JSON object conforming to this schema:
{schema_str}

Rules:
- Output ONLY the JSON object
- Every field in "required" must be present
- Use empty arrays if no entities of that type exist
- For "type" fields, use only the allowed enum values

Text: {text}"""

    response = query_llm("You are a structured data extraction system. Output only valid JSON.", prompt)

    # Clean up
    response = response.strip()
    if response.startswith("```"):
        response = response.split("\n", 1)[1]
        response = response.rsplit("```", 1)[0]
    response = response.strip()

    return json.loads(response)

test_text = """On March 15, 2024, Satya Nadella announced that Microsoft would partner
with OpenAI to build a new research lab in London. The project, backed by
the Gates Foundation, aims to advance AI safety research by 2026."""

result = extract_with_schema(test_text)
print("Schema-constrained extraction:")
print(json.dumps(result, indent=2))

In [ ]:
#@title 🎧 What to Look For: Compare Approaches Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_compare_approaches_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Comparing the Three Approaches

In [ ]:
# Visualization: Comparing the three approaches
fig, ax = plt.subplots(figsize=(10, 6))

approaches = ['Format\nInstructions', 'XML\nTags', 'Schema\nin Prompt']
metrics = {
    'Parse Reliability': [0.82, 0.90, 0.95],
    'Schema Compliance': [0.70, 0.78, 0.93],
    'Ease of Implementation': [0.95, 0.80, 0.65],
}

x = np.arange(len(approaches))
width = 0.25
multiplier = 0

colors = ['#2196F3', '#4CAF50', '#FF9800']
for i, (metric, values) in enumerate(metrics.items()):
    offset = width * multiplier
    bars = ax.bar(x + offset, values, width, label=metric, color=colors[i],
                  edgecolor='black', linewidth=1)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.0%}', ha='center', fontsize=9, fontweight='bold')
    multiplier += 1

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Three Approaches to Structured Output: Trade-offs', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(approaches, fontsize=11)
ax.legend(loc='upper left', fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key trade-off: Schema-in-prompt gives the best reliability")
print("but requires the most setup. Format instructions are simplest")
print("but least reliable. Choose based on your use case.")

In [ ]:
#@title 🎧 Transition: Building Chain Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_15_building_chain_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Building a Prompt Chain With Gate Checks

Now let us put structured output and chaining together. We will build a 3-step document analysis pipeline where each step produces structured output and a validation gate checks the result before passing it to the next step.

### Step 1: Entity Extraction

In [ ]:
#@title 🎧 Code Walkthrough: Step1 Entity Extraction
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_16_step1_entity_extraction.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def step1_extract_entities(document):
    """Step 1: Extract entities from the document as structured JSON."""
    prompt = f"""Extract all named entities from the document below.

Return ONLY a JSON object with this exact structure:
{{
    "entities": [
        {{"text": "entity name", "type": "PERSON|ORG|LOCATION|DATE|PRODUCT", "context": "brief phrase showing how it appears"}}
    ],
    "entity_count": <integer>,
    "document_language": "en"
}}

Rules:
- Include every entity, even if it appears multiple times
- "context" should be a short phrase (5-10 words) from the document
- entity_count must match the length of the entities array

Document:
{document}"""

    response = query_llm(
        "You are a named entity recognition system. Output only valid JSON.",
        prompt
    )

    # Clean response
    response = response.strip()
    if response.startswith("```"):
        response = response.split("\n", 1)[1]
        response = response.rsplit("```", 1)[0].strip()

    return json.loads(response)

# Test Step 1
document = """Apple CEO Tim Cook unveiled the Vision Pro headset at WWDC 2023 in
Cupertino, California. The device, priced at $3,499, will compete with
Meta's Quest 3 headset. Microsoft CEO Satya Nadella commented that the
mixed reality market is "entering a new era." Analysts at Goldman Sachs
predict the AR/VR market will reach $80 billion by 2025."""

step1_result = step1_extract_entities(document)
print("Step 1 — Entity Extraction:")
print(json.dumps(step1_result, indent=2))

In [ ]:
#@title 🎧 Code Walkthrough: Gate1 Validate Entities
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_17_gate1_validate_entities.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Gate Check 1: Validate Entity Extraction

In [ ]:
def gate_check_entities(result):
    """Validate the output of Step 1 before passing to Step 2."""
    errors = []

    # Check required keys
    required_keys = ["entities", "entity_count", "document_language"]
    for key in required_keys:
        if key not in result:
            errors.append(f"Missing required key: {key}")

    # Check entity_count consistency
    if "entities" in result and "entity_count" in result:
        actual_count = len(result["entities"])
        claimed_count = result["entity_count"]
        if actual_count != claimed_count:
            errors.append(f"entity_count mismatch: claimed {claimed_count}, actual {actual_count}")
            # Auto-fix: use actual count
            result["entity_count"] = actual_count

    # Check that each entity has required fields
    if "entities" in result:
        valid_types = {"PERSON", "ORG", "LOCATION", "DATE", "PRODUCT"}
        for i, entity in enumerate(result["entities"]):
            for field in ["text", "type", "context"]:
                if field not in entity:
                    errors.append(f"Entity {i} missing field: {field}")
            if entity.get("type") not in valid_types:
                errors.append(f"Entity {i} has invalid type: {entity.get('type')}")

    # Check minimum entity count (a real document should have some entities)
    if "entities" in result and len(result["entities"]) == 0:
        errors.append("No entities found — possible extraction failure")

    passed = len(errors) == 0
    return passed, errors, result

passed, errors, validated_result = gate_check_entities(step1_result)
print(f"\nGate Check 1: {'PASSED' if passed else 'FAILED'}")
if errors:
    for err in errors:
        print(f"  Warning: {err}")
else:
    print(f"  All checks passed. {validated_result['entity_count']} entities validated.")

In [ ]:
#@title 🎧 Code Walkthrough: Step2 Claim Classification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_18_step2_claim_classification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Step 2: Claim Classification

In [ ]:
def step2_classify_claims(document, entities):
    """Step 2: Identify and classify claims in the document."""
    entity_list = ", ".join([e["text"] for e in entities["entities"]])

    prompt = f"""Analyze the document below and identify all factual claims.
The document mentions these entities: {entity_list}

Return ONLY a JSON object with this structure:
{{
    "claims": [
        {{
            "claim": "the factual claim as stated",
            "type": "FACTUAL|OPINION|PREDICTION|COMPARISON",
            "confidence": "HIGH|MEDIUM|LOW",
            "entities_involved": ["entity1", "entity2"],
            "verifiable": true or false
        }}
    ],
    "total_claims": <integer>,
    "factual_ratio": <float between 0 and 1>
}}

Claim types:
- FACTUAL: A verifiable statement of fact ("Apple launched X on date Y")
- OPINION: A subjective judgment ("the market is exciting")
- PREDICTION: A forward-looking statement ("will reach $80 billion")
- COMPARISON: Compares two or more things ("will compete with Meta")

Document:
{document}"""

    response = query_llm(
        "You are a claim analysis system. Output only valid JSON.",
        prompt
    )

    response = response.strip()
    if response.startswith("```"):
        response = response.split("\n", 1)[1]
        response = response.rsplit("```", 1)[0].strip()

    return json.loads(response)

step2_result = step2_classify_claims(document, step1_result)
print("Step 2 — Claim Classification:")
print(json.dumps(step2_result, indent=2))

In [ ]:
#@title 🎧 Code Walkthrough: Gate2 Validate Claims
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_19_gate2_validate_claims.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Gate Check 2: Validate Claim Classification

In [ ]:
def gate_check_claims(result, entities_result):
    """Validate Step 2 output before passing to Step 3."""
    errors = []

    # Check required keys
    for key in ["claims", "total_claims", "factual_ratio"]:
        if key not in result:
            errors.append(f"Missing required key: {key}")

    if "claims" in result:
        valid_types = {"FACTUAL", "OPINION", "PREDICTION", "COMPARISON"}
        valid_confidence = {"HIGH", "MEDIUM", "LOW"}
        known_entities = {e["text"] for e in entities_result["entities"]}

        for i, claim in enumerate(result["claims"]):
            # Check required fields
            for field in ["claim", "type", "confidence", "entities_involved", "verifiable"]:
                if field not in claim:
                    errors.append(f"Claim {i} missing field: {field}")

            # Check valid enum values
            if claim.get("type") not in valid_types:
                errors.append(f"Claim {i} has invalid type: {claim.get('type')}")
            if claim.get("confidence") not in valid_confidence:
                errors.append(f"Claim {i} has invalid confidence: {claim.get('confidence')}")

        # Check total_claims consistency
        if "total_claims" in result:
            if result["total_claims"] != len(result["claims"]):
                errors.append(f"total_claims mismatch: {result['total_claims']} vs {len(result['claims'])}")
                result["total_claims"] = len(result["claims"])

        # Check factual_ratio is valid
        if "factual_ratio" in result:
            if not (0 <= result["factual_ratio"] <= 1):
                errors.append(f"factual_ratio out of range: {result['factual_ratio']}")

    passed = len(errors) == 0
    return passed, errors, result

passed, errors, validated_claims = gate_check_claims(step2_result, step1_result)
print(f"\nGate Check 2: {'PASSED' if passed else 'FAILED'}")
if errors:
    for err in errors:
        print(f"  Warning: {err}")
else:
    print(f"  All checks passed. {validated_claims['total_claims']} claims validated.")

In [ ]:
#@title 🎧 Code Walkthrough: Step3 Synthesis Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_20_step3_synthesis_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Step 3: Synthesis Summary

In [ ]:
def step3_synthesize(document, entities, claims):
    """Step 3: Produce a structured summary combining entity and claim analysis."""
    prompt = f"""You are given a document and its analysis. Produce a structured summary.

Entity analysis (from Step 1):
{json.dumps(entities, indent=2)}

Claim analysis (from Step 2):
{json.dumps(claims, indent=2)}

Return ONLY a JSON object with this structure:
{{
    "title": "A concise title for the document (under 10 words)",
    "summary": "A 2-3 sentence summary of the document",
    "key_players": [
        {{"name": "entity name", "role": "their role in the story"}}
    ],
    "main_claims": [
        {{"claim": "the claim", "assessment": "brief assessment of reliability"}}
    ],
    "overall_sentiment": "positive|negative|neutral|mixed",
    "confidence_score": <float between 0 and 1>
}}

Focus on the most important entities and claims. Limit key_players to 3 and main_claims to 3.

Original document:
{document}"""

    response = query_llm(
        "You are a document summarization system. Output only valid JSON.",
        prompt
    )

    response = response.strip()
    if response.startswith("```"):
        response = response.split("\n", 1)[1]
        response = response.rsplit("```", 1)[0].strip()

    return json.loads(response)

step3_result = step3_synthesize(document, step1_result, step2_result)
print("Step 3 — Synthesis Summary:")
print(json.dumps(step3_result, indent=2))

In [ ]:
#@title 🎧 Code Walkthrough: Complete Pipeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_21_complete_pipeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### The Complete Pipeline With Retry Logic

In [ ]:
def run_analysis_pipeline(document, max_retries=2):
    """
    Run the complete 3-step analysis pipeline with gate checks and retries.

    Returns the final synthesis or an error report.
    """
    pipeline_log = {"steps_completed": 0, "retries": 0, "errors": []}

    # --- Step 1: Entity Extraction ---
    print("Running Step 1: Entity Extraction...")
    for attempt in range(max_retries + 1):
        try:
            step1 = step1_extract_entities(document)
            passed, errors, step1 = gate_check_entities(step1)
            if passed:
                print(f"  Step 1 PASSED ({step1['entity_count']} entities)")
                pipeline_log["steps_completed"] = 1
                break
            else:
                print(f"  Step 1 gate check failed (attempt {attempt + 1}): {errors}")
                pipeline_log["retries"] += 1
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  Step 1 parse error (attempt {attempt + 1}): {e}")
            pipeline_log["retries"] += 1
    else:
        pipeline_log["errors"].append("Step 1 failed after all retries")
        return None, pipeline_log

    # --- Step 2: Claim Classification ---
    print("Running Step 2: Claim Classification...")
    for attempt in range(max_retries + 1):
        try:
            step2 = step2_classify_claims(document, step1)
            passed, errors, step2 = gate_check_claims(step2, step1)
            if passed:
                print(f"  Step 2 PASSED ({step2['total_claims']} claims)")
                pipeline_log["steps_completed"] = 2
                break
            else:
                print(f"  Step 2 gate check failed (attempt {attempt + 1}): {errors}")
                pipeline_log["retries"] += 1
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  Step 2 parse error (attempt {attempt + 1}): {e}")
            pipeline_log["retries"] += 1
    else:
        pipeline_log["errors"].append("Step 2 failed after all retries")
        return None, pipeline_log

    # --- Step 3: Synthesis ---
    print("Running Step 3: Synthesis...")
    for attempt in range(max_retries + 1):
        try:
            step3 = step3_synthesize(document, step1, step2)
            print(f"  Step 3 PASSED")
            pipeline_log["steps_completed"] = 3
            return step3, pipeline_log
        except (json.JSONDecodeError, KeyError) as e:
            print(f"  Step 3 parse error (attempt {attempt + 1}): {e}")
            pipeline_log["retries"] += 1

    pipeline_log["errors"].append("Step 3 failed after all retries")
    return None, pipeline_log

# Run the full pipeline
print("=" * 60)
print("  DOCUMENT ANALYSIS PIPELINE")
print("=" * 60)
print()

final_result, log = run_analysis_pipeline(document)

print("\n" + "=" * 60)
if final_result:
    print("PIPELINE RESULT:")
    print(json.dumps(final_result, indent=2))
else:
    print("PIPELINE FAILED")

print(f"\nPipeline log: {json.dumps(log, indent=2)}")

In [ ]:
#@title 🎧 Transition: Putting It Together Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_22_putting_it_together_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Putting It All Together

Let us now visualize how the gate check mechanism improves pipeline reliability. We will also build a reusable chain architecture.

In [ ]:
#@title 🎧 What to Look For: Flow Diagram Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_23_flow_diagram_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualization: Gate check flow diagram (simulated with matplotlib)
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('Prompt Chain With Gate Checks', fontsize=16, fontweight='bold', pad=20)

# Draw the pipeline steps
step_positions = [(1, 3), (4.5, 3), (8, 3), (11.5, 3)]
step_labels = ['Step 1:\nEntity\nExtraction', 'Gate\nCheck 1', 'Step 2:\nClaim\nClassification', 'Gate\nCheck 2']
step_colors = ['#2196F3', '#FF9800', '#2196F3', '#FF9800']

for (x, y), label, color in zip(step_positions, step_labels, step_colors):
    box = plt.Rectangle((x - 0.9, y - 0.8), 1.8, 1.6,
                         facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')

# Add Step 3 and final output
for (x, y), label, color in [((8, 3), 'Step 2:\nClaim\nClassification', '#2196F3')]:
    pass  # Already drawn

# Step 3
box = plt.Rectangle((11.5 - 0.9, 3 - 0.8), 1.8, 1.6,
                     facecolor='#FF9800', edgecolor='black', linewidth=2, alpha=0.8)
ax.add_patch(box)
ax.text(11.5, 3, 'Gate\nCheck 2', ha='center', va='center', fontsize=9,
        fontweight='bold', color='white')

# Arrows between steps (forward flow)
arrow_style = dict(arrowstyle='->', color='black', lw=2)
ax.annotate('', xy=(3.6, 3), xytext=(1.9, 3), arrowprops=arrow_style)
ax.annotate('', xy=(7.1, 3), xytext=(5.4, 3), arrowprops=arrow_style)
ax.annotate('', xy=(10.6, 3), xytext=(8.9, 3), arrowprops=arrow_style)

# Pass labels
ax.text(2.75, 3.3, 'JSON', ha='center', fontsize=8, color='green', fontstyle='italic')
ax.text(6.25, 3.3, 'Pass', ha='center', fontsize=8, color='green', fontweight='bold')
ax.text(9.75, 3.3, 'JSON', ha='center', fontsize=8, color='green', fontstyle='italic')

# Retry arrows (curved, going back)
ax.annotate('', xy=(3.6, 2.0), xytext=(4.5, 1.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5,
                          connectionstyle='arc3,rad=0.3'))
ax.text(3.5, 1.3, 'Retry', ha='center', fontsize=8, color='red', fontweight='bold')

ax.annotate('', xy=(7.1, 2.0), xytext=(8.0, 1.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5,
                          connectionstyle='arc3,rad=0.3'))
ax.text(7.0, 1.3, 'Retry', ha='center', fontsize=8, color='red', fontweight='bold')

# Final output box
box_final = plt.Rectangle((12.8 - 0.7, 3 - 0.6), 1.4, 1.2,
                           facecolor='#4CAF50', edgecolor='black', linewidth=2, alpha=0.9)
ax.add_patch(box_final)

# Add Step 3 box between Gate 2 and Final
box_s3 = plt.Rectangle((11.5 - 0.9, 3 - 0.8), 1.8, 1.6,
                        facecolor='#FF9800', edgecolor='black', linewidth=2, alpha=0.8)

# Actually, let us simplify the diagram
ax.text(13.1, 3, 'Final\nOutput', ha='center', va='center', fontsize=9,
        fontweight='bold', color='white')

ax.annotate('', xy=(12.4, 3), xytext=(12.4, 3), arrowprops=arrow_style)

# Add a legend
legend_items = [
    plt.Rectangle((0, 0), 1, 1, facecolor='#2196F3', alpha=0.8),
    plt.Rectangle((0, 0), 1, 1, facecolor='#FF9800', alpha=0.8),
    plt.Rectangle((0, 0), 1, 1, facecolor='#4CAF50', alpha=0.8),
]
ax.legend(legend_items, ['LLM Step', 'Gate Check', 'Final Output'],
          loc='upper right', fontsize=10)

# Add title annotation
ax.text(7, 5.3, 'Each gate validates structure + content before passing data forward',
        ha='center', fontsize=11, fontstyle='italic', color='gray')
ax.text(7, 0.5, 'Failed gates trigger retries with the same input — preventing error propagation',
        ha='center', fontsize=11, fontstyle='italic', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 What to Look For: Reliability Comparison Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_24_reliability_comparison_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualization: Reliability comparison — ungated vs gated chains
fig, ax = plt.subplots(figsize=(10, 6))

step_counts = range(1, 8)
ungated_90 = [0.90 ** n for n in step_counts]
ungated_95 = [0.95 ** n for n in step_counts]
gated_90 = [0.98 ** n for n in step_counts]   # 90% + 80% gate = 98%
gated_95 = [0.99 ** n for n in step_counts]   # 95% + 80% gate = 99%

ax.plot(step_counts, ungated_90, 'o--', color='#F44336', linewidth=2,
        label='Ungated (p=90%)', markersize=8)
ax.plot(step_counts, ungated_95, 's--', color='#FF9800', linewidth=2,
        label='Ungated (p=95%)', markersize=8)
ax.plot(step_counts, gated_90, 'o-', color='#4CAF50', linewidth=2.5,
        label='Gated (p=90%, gate=80%)', markersize=8)
ax.plot(step_counts, gated_95, 's-', color='#2196F3', linewidth=2.5,
        label='Gated (p=95%, gate=80%)', markersize=8)

ax.axhline(y=0.80, color='gray', linestyle=':', alpha=0.5, label='80% threshold')
ax.fill_between(step_counts, 0, 0.80, alpha=0.05, color='red')
ax.fill_between(step_counts, 0.80, 1.0, alpha=0.05, color='green')

ax.set_xlabel('Number of Chain Steps', fontsize=13)
ax.set_ylabel('End-to-End Chain Accuracy', fontsize=13)
ax.set_title('Gate Checks Dramatically Improve Chain Reliability', fontsize=14)
ax.legend(fontsize=10, loc='lower left')
ax.set_ylim(0, 1.05)
ax.set_xticks(list(step_counts))
ax.grid(True, alpha=0.3)

# Annotate the gap at 4 steps
ax.annotate('', xy=(4, 0.98**4), xytext=(4, 0.90**4),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
ax.text(4.3, 0.83, f'Gate boost:\n{0.98**4:.1%} vs {0.90**4:.1%}',
        fontsize=10, color='purple', fontweight='bold')

plt.tight_layout()
plt.show()

print("With an 80% error-catching gate, a 90% step becomes effectively 98%.")
print("Over 4 steps: 65.6% (ungated) vs 92.2% (gated).")
print("Gate checks are the difference between a toy and a production system.")

In [ ]:
#@title 🎧 Transition: Advanced Patterns Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_25_advanced_patterns_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Advanced Patterns: Robust Chain Design

Let us now look at patterns that make chains even more reliable in practice.

### Pattern 1: Structured Output With Fallback Parsing

In [ ]:
#@title 🎧 Code Walkthrough: Fallback Parsing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_26_fallback_parsing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def robust_json_parse(response, expected_keys=None):
    """
    Parse JSON from an LLM response with multiple fallback strategies.

    Strategy 1: Direct parse
    Strategy 2: Strip markdown fences
    Strategy 3: Find JSON object in text
    Strategy 4: Find JSON array in text
    """
    strategies = []

    # Strategy 1: Direct parse
    try:
        result = json.loads(response.strip())
        strategies.append(("direct", True))
        if expected_keys and not all(k in result for k in expected_keys):
            strategies[-1] = ("direct", "missing_keys")
        else:
            return result, strategies
    except json.JSONDecodeError:
        strategies.append(("direct", False))

    # Strategy 2: Strip markdown fences
    cleaned = response.strip()
    if "```" in cleaned:
        # Remove markdown code fences
        lines = cleaned.split("\n")
        json_lines = []
        inside_fence = False
        for line in lines:
            if line.strip().startswith("```"):
                inside_fence = not inside_fence
                continue
            if inside_fence or not any(line.strip().startswith(c) for c in ["Sure", "Here", "The", "I "]):
                json_lines.append(line)
        cleaned = "\n".join(json_lines).strip()

        try:
            result = json.loads(cleaned)
            strategies.append(("strip_fences", True))
            return result, strategies
        except json.JSONDecodeError:
            strategies.append(("strip_fences", False))

    # Strategy 3: Find JSON object with regex
    import re
    json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response, re.DOTALL)
    if json_match:
        try:
            result = json.loads(json_match.group())
            strategies.append(("regex_object", True))
            return result, strategies
        except json.JSONDecodeError:
            strategies.append(("regex_object", False))

    # All strategies failed
    return None, strategies

# Test the robust parser
messy_responses = [
    '{"name": "Alice", "age": 30}',                          # Clean
    'Here is the JSON:\n```json\n{"name": "Bob"}\n```',       # Markdown fence
    'Sure! {"name": "Charlie", "age": 25} Hope that helps!',  # Surrounded by text
]

for resp in messy_responses:
    result, strategies = robust_json_parse(resp)
    succeeded = [s for s, ok in strategies if ok is True]
    print(f"Input: {resp[:50]}...")
    print(f"  Parsed: {result}")
    print(f"  Strategy: {succeeded[-1] if succeeded else 'FAILED'}")
    print()

In [ ]:
#@title 🎧 Code Walkthrough: Reusable Chain Step
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_27_reusable_chain_step.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Pattern 2: Chain Step With Built-in Validation

In [ ]:
def chain_step(prompt, system_prompt, expected_keys, step_name, max_retries=2):
    """
    A reusable chain step with built-in JSON parsing, validation, and retry.

    This is the building block for any prompt chain.
    """
    for attempt in range(max_retries + 1):
        try:
            response = query_llm(system_prompt, prompt)
            result, strategies = robust_json_parse(response, expected_keys)

            if result is None:
                print(f"  [{step_name}] Attempt {attempt + 1}: JSON parse failed")
                continue

            # Check required keys
            missing = [k for k in expected_keys if k not in result]
            if missing:
                print(f"  [{step_name}] Attempt {attempt + 1}: Missing keys: {missing}")
                continue

            print(f"  [{step_name}] Success on attempt {attempt + 1}")
            return result

        except Exception as e:
            print(f"  [{step_name}] Attempt {attempt + 1}: Error — {e}")

    print(f"  [{step_name}] FAILED after {max_retries + 1} attempts")
    return None

# Test the reusable chain step
result = chain_step(
    prompt='Extract the topic and sentiment from: "The new AI regulations are a step in the right direction." Return JSON with keys: topic, sentiment',
    system_prompt="Output only valid JSON.",
    expected_keys=["topic", "sentiment"],
    step_name="Quick Extract"
)
print(f"\nResult: {result}")

## 8. Your Turn


In [ ]:
#@title 🎧 Before You Start: Todo1 Pipeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_28_todo1_pipeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 1: Build a 3-Step Document Analysis Pipeline With Gates

Build a pipeline that processes a news article in three steps:

1. **Extract**: Pull out key facts (who, what, when, where, why)
2. **Classify**: Categorize the article (politics, technology, business, science, sports) and assess bias
3. **Summarize**: Produce a structured briefing with key takeaways

Each step must produce valid JSON and pass through a validation gate before the next step runs.

In [ ]:
def todo_step1_extract_facts(article):
    """
    Step 1: Extract the 5 W's from a news article.

    Must return JSON with keys:
    - who: list of people/organizations involved
    - what: what happened (1-2 sentences)
    - when: when it happened
    - where: where it happened
    - why: why it matters (1-2 sentences)
    """
    # ============ TODO ============
    # Write a prompt that extracts the 5 W's as structured JSON.
    # Use the schema-in-prompt approach for best reliability.
    # ==============================

    prompt = "???"  # YOUR PROMPT HERE
    pass

def todo_gate1_validate_facts(result):
    """
    Gate 1: Validate the extracted facts.

    Check:
    - All 5 keys (who, what, when, where, why) are present
    - "who" is a non-empty list
    - "what" is a non-empty string
    """
    # ============ TODO ============
    # Implement validation logic
    # Return (passed: bool, errors: list, result: dict)
    # ==============================

    pass

def todo_step2_classify(article, facts):
    """
    Step 2: Classify the article category and assess bias.

    Must return JSON with keys:
    - category: one of [politics, technology, business, science, sports, other]
    - bias: one of [left, center-left, center, center-right, right, unclear]
    - bias_indicators: list of specific phrases that indicate bias
    - objectivity_score: float 0-1
    """
    # ============ TODO ============
    # Write a prompt that classifies the article.
    # Pass the extracted facts from Step 1 as context.
    # ==============================

    prompt = "???"  # YOUR PROMPT HERE
    pass

def todo_step3_summarize(article, facts, classification):
    """
    Step 3: Produce a structured briefing.

    Must return JSON with keys:
    - headline: a neutral, factual headline (under 15 words)
    - briefing: 3-bullet executive summary
    - key_takeaways: list of 2-3 takeaways
    - follow_up_questions: list of 2 questions a reader might ask next
    """
    # ============ TODO ============
    # Write a prompt that synthesizes facts + classification into a briefing.
    # ==============================

    prompt = "???"  # YOUR PROMPT HERE
    pass

# Test article for your pipeline
test_article = """
The European Union announced sweeping new regulations for artificial intelligence
on Wednesday, requiring companies like Google, Microsoft, and OpenAI to disclose
training data sources and implement safety testing before deploying AI systems in
the EU market. The AI Act, which took three years to negotiate, imposes fines of
up to 7% of global revenue for violations. Industry groups criticized the rules
as overly burdensome, while digital rights organizations praised them as a
necessary step toward accountability. EU Commissioner Thierry Breton called it
"a landmark moment for democratic governance of technology."
"""

# Uncomment and run after implementing:
# step1 = todo_step1_extract_facts(test_article)
# passed, errors, step1 = todo_gate1_validate_facts(step1)
# if passed:
#     step2 = todo_step2_classify(test_article, step1)
#     step3 = todo_step3_summarize(test_article, step1, step2)
#     print(json.dumps(step3, indent=2))

In [ ]:
#@title 🎧 Before You Start: Todo2 Extractor
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_29_todo2_extractor.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Build a Structured Data Extractor With Schema Validation

Build a reusable function that takes an arbitrary JSON schema and a text input, then extracts structured data matching that schema — with validation.

In [ ]:
def todo_structured_extractor(text, schema, max_retries=2):
    """
    Extract structured data from text according to a given JSON schema.

    Args:
        text: The input text to extract from
        schema: A JSON schema dict describing the desired output
        max_retries: Number of retry attempts on validation failure

    Returns:
        (result_dict, validation_report)

    Your implementation should:
    1. Build a prompt that includes the schema
    2. Call the LLM and parse the response as JSON
    3. Validate the result against the schema (check required fields, types, enums)
    4. Retry on failure
    5. Return the result and a validation report
    """
    # ============ TODO ============
    # Implement the structured extractor.
    #
    # Hints:
    # - Convert the schema to a string and embed it in the prompt
    # - Use robust_json_parse() for parsing
    # - Write a validate_against_schema() helper function
    # - Track which attempt succeeded
    # ==============================

    pass

# Test with a product review schema
review_schema = {
    "type": "object",
    "required": ["product_name", "rating", "pros", "cons", "recommendation"],
    "properties": {
        "product_name": {"type": "string"},
        "rating": {"type": "number", "minimum": 1, "maximum": 5},
        "pros": {"type": "array", "items": {"type": "string"}},
        "cons": {"type": "array", "items": {"type": "string"}},
        "recommendation": {"type": "string", "enum": ["buy", "skip", "wait_for_sale"]}
    }
}

review_text = """I bought the Sony WH-1000XM5 headphones last month. The noise
cancellation is absolutely incredible — best I have ever experienced. Sound quality
is rich and detailed across all frequencies. Battery life easily lasts 30 hours.
However, the folding mechanism is gone (they do not fold flat anymore), and at $400
they are pricey compared to alternatives. The touch controls can be finicky in cold
weather. Overall, if noise cancellation is your priority, these are unbeatable, but
wait for a sale if you can."""

# Uncomment after implementing:
# result, report = todo_structured_extractor(review_text, review_schema)
# print(json.dumps(result, indent=2))
# print(f"\nValidation report: {report}")

In [ ]:
#@title 🎧 Wrap-Up: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_30_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### Reflection Questions
1. Why is the **schema-in-prompt** approach more reliable than simply asking for "JSON output"? What information does the schema convey that free-form instructions do not?
2. In our error propagation formula $P(\text{chain}) = \prod p_i$, we assumed independence between steps. In practice, is this assumption valid? When might errors in Step 1 **increase** the probability of errors in Step 2?
3. You have a 5-step pipeline where one step has 80% accuracy and the rest are at 98%. Is it better to add a gate check to the weak step or to split it into two simpler sub-steps? What are the trade-offs?

### Key Takeaways

1. **Structured output is a contract**: JSON, XML, and schemas are not formatting preferences — they are machine-to-machine communication protocols that make LLM output programmatically usable.
2. **Chains multiply errors**: $P(\text{chain}) = \prod p_i$ means that even 95% per-step accuracy gives only 81.5% over 4 steps. Always calculate your end-to-end reliability.
3. **Gate checks are essential**: A gate that catches 80% of errors can boost a 90% step to an effective 98%, transforming a fragile chain into a robust pipeline.
4. **Three approaches, one goal**: Format instructions are simplest, XML tags are most flexible, and schema-in-prompt is most reliable. Choose based on your needs.

### Optional Challenges
1. **Retry With Rephrasing**: Modify the `chain_step` function so that on retry, it appends the previous failed output and an error message to the prompt, asking the LLM to fix the specific issue.
2. **Parallel Chains With Voting**: Run the same extraction step 3 times in parallel and use majority voting to pick the most common answer. Measure how this affects accuracy.
3. **Dynamic Schema Generation**: Write a function that takes a sample JSON object and automatically generates a schema for it, then uses that schema in extraction prompts.